In [0]:
ctx = spark.sql("""
SELECT
    current_catalog() AS catalog,
    current_schema() AS schema
""").first()

CATALOG = ctx["catalog"]
SCHEMA = ctx["schema"]

print("Catalog :", CATALOG)
print("Schema  :", SCHEMA)
print("Full namespace:", f"{CATALOG}.{SCHEMA}")

Catalog : workspace
Schema  : default
Full namespace: workspace.default


In [0]:
%sh
curl -s "https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000" | head -c 1000

[{"page":1,"pages":1,"per_page":1000,"total":330,"sourceid":"2","lastupdated":"2026-07-13"},[{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"CN","value":"China"},"countryiso3code":"CHN","date":"2025","value":1406585000,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"CN","value":"China"},"countryiso3code":"CHN","date":"2024","value":1408975000,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"CN","value":"China"},"countryiso3code":"CHN","date":"2023","value":1410710000,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"CN","value":"China"},"countryiso3code":"CHN","date":"2022","value":1412175000,"unit":"","obs_status":"","decimal":0},{"indicator":{"id":"SP.POP.TOTL","value":"Population, total"},"country":{"id":"CN","value":"China"},"countryiso3code":"CHN"

In [0]:
import requests

url = (
    "https://api.worldbank.org/v2/"
    "country/IND;USA;CHN;GBR;DEU/"
    "indicator/SP.POP.TOTL"
    "?format=json&per_page=1000"
)

response = requests.get(url, timeout=30)

print("HTTP status:", response.status_code)

response.raise_for_status()

payload = response.json()

print("Metadata:")
print(payload[0])

print("\nNumber of records:")
print(len(payload[1]))

HTTP status: 200
Metadata:
{'page': 1, 'pages': 1, 'per_page': 1000, 'total': 330, 'sourceid': '2', 'lastupdated': '2026-07-13'}

Number of records:
330


In [0]:
payload[1][0]

{'indicator': {'id': 'SP.POP.TOTL', 'value': 'Population, total'},
 'country': {'id': 'CN', 'value': 'China'},
 'countryiso3code': 'CHN',
 'date': '2025',
 'value': 1406585000,
 'unit': '',
 'obs_status': '',
 'decimal': 0}

In [0]:
import json
from pyspark.sql import functions as F

records = payload[1]

rows = [
    (json.dumps(record), url)
    for record in records
]

landing_df = (
    spark.createDataFrame(
        rows,
        ["raw_json", "source_url"]
    )
    .withColumn("ingested_at", F.current_timestamp())
)

display(landing_df.limit(10))

raw_json,source_url,ingested_at
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2025"", ""value"": 1406585000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2024"", ""value"": 1408975000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2023"", ""value"": 1410710000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2022"", ""value"": 1412175000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2021"", ""value"": 1412360000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2020"", ""value"": 1411100000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2019"", ""value"": 1407745000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2018"", ""value"": 1402760000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2017"", ""value"": 1396215000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2016"", ""value"": 1387790000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:05.281Z


In [0]:
ICEBERG_TABLE = f"{CATALOG}.{SCHEMA}.wb_population_landing_iceberg"

(
    landing_df.write
    .format("iceberg")
    .mode("overwrite")
    .saveAsTable(ICEBERG_TABLE)
)

print("Created:", ICEBERG_TABLE)

Created: workspace.default.wb_population_landing_iceberg


In [0]:
display(spark.table(ICEBERG_TABLE).limit(10))

raw_json,source_url,ingested_at
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2025"", ""value"": 1406585000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2024"", ""value"": 1408975000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2023"", ""value"": 1410710000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2022"", ""value"": 1412175000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2021"", ""value"": 1412360000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2020"", ""value"": 1411100000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2019"", ""value"": 1407745000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2018"", ""value"": 1402760000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2017"", ""value"": 1396215000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2016"", ""value"": 1387790000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z


In [0]:
spark.sql(f"DESCRIBE DETAIL {ICEBERG_TABLE}").show(truncate=False)

+-------+------------------------------------+-----------------------------------------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------+-------------------+----------------+-----------------+--------+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.population_bronze"

bronze_df = spark.table(ICEBERG_TABLE)

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(BRONZE_TABLE)
)

print("Created:", BRONZE_TABLE)

Created: workspace.default.population_bronze


In [0]:
display(spark.table(BRONZE_TABLE).limit(10))

raw_json,source_url,ingested_at
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2025"", ""value"": 1406585000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2024"", ""value"": 1408975000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2023"", ""value"": 1410710000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2022"", ""value"": 1412175000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2021"", ""value"": 1412360000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2020"", ""value"": 1411100000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2019"", ""value"": 1407745000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2018"", ""value"": 1402760000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2017"", ""value"": 1396215000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z
"{""indicator"": {""id"": ""SP.POP.TOTL"", ""value"": ""Population, total""}, ""country"": {""id"": ""CN"", ""value"": ""China""}, ""countryiso3code"": ""CHN"", ""date"": ""2016"", ""value"": 1387790000, ""unit"": """", ""obs_status"": """", ""decimal"": 0}",https://api.worldbank.org/v2/country/IND;USA;CHN;GBR;DEU/indicator/SP.POP.TOTL?format=json&per_page=1000,2026-08-26T19:06:31.953Z


In [0]:
bronze = spark.table(BRONZE_TABLE)

silver_df = (
    bronze
    .select(
        F.get_json_object("raw_json", "$.countryiso3code")
            .alias("country_code"),

        F.get_json_object("raw_json", "$.country.value")
            .alias("country_name"),

        F.get_json_object("raw_json", "$.indicator.id")
            .alias("indicator_code"),

        F.get_json_object("raw_json", "$.indicator.value")
            .alias("indicator_name"),

        F.get_json_object("raw_json", "$.date")
            .cast("int")
            .alias("year"),

        F.get_json_object("raw_json", "$.value")
            .cast("long")
            .alias("population"),

        "ingested_at"
    )
    .filter(F.col("country_code").isNotNull())
    .filter(F.col("year").isNotNull())
    .filter(F.col("population").isNotNull())
)

display(silver_df.limit(20))

country_code,country_name,indicator_code,indicator_name,year,population,ingested_at
CHN,China,SP.POP.TOTL,"Population, total",2025,1406585000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2024,1408975000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2023,1410710000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2022,1412175000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2021,1412360000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2020,1411100000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2019,1407745000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2018,1402760000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2017,1396215000,2026-08-26T19:06:31.953Z
CHN,China,SP.POP.TOTL,"Population, total",2016,1387790000,2026-08-26T19:06:31.953Z


In [0]:
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.population_silver"

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER_TABLE)
)

print("Created:", SILVER_TABLE)

Created: workspace.default.population_silver


In [0]:
silver = spark.table(SILVER_TABLE)

print("Rows:", silver.count())

silver.select(
    F.min("year").alias("oldest_year"),
    F.max("year").alias("latest_year"),
    F.countDistinct("country_code").alias("countries")
).show()

Rows: 330
+-----------+-----------+---------+
|oldest_year|latest_year|countries|
+-----------+-----------+---------+
|       1960|       2025|        5|
+-----------+-----------+---------+



In [0]:
silver.groupBy("country_name") \
      .count() \
      .orderBy("country_name") \
      .show()

+--------------+-----+
|  country_name|count|
+--------------+-----+
|         China|   66|
|       Germany|   66|
|         India|   66|
|United Kingdom|   66|
| United States|   66|
+--------------+-----+



In [0]:
silver = spark.table(SILVER_TABLE)

gold_df = (
    silver
    .withColumn(
        "decade",
        (F.floor(F.col("year") / 10) * 10).cast("int")
    )
    .groupBy(
        "country_code",
        "country_name",
        "decade"
    )
    .agg(
        F.round(F.avg("population"), 0)
            .cast("long")
            .alias("avg_population"),

        F.min("population")
            .alias("min_population"),

        F.max("population")
            .alias("max_population"),

        F.count("*")
            .alias("years_available")
    )
    .orderBy("country_name", "decade")
)

display(gold_df)

country_code,country_name,decade,avg_population,min_population,max_population,years_available
CHN,China,1960,714953000,660330000,796025000,10
CHN,China,1970,901944500,818315000,969005000,10
CHN,China,1980,1046603000,981235000,1118650000,10
CHN,China,1990,1196836000,1135185000,1252735000,10
CHN,China,2000,1298791000,1262645000,1331260000,10
CHN,China,2010,1374640000,1337705000,1407745000,10
CHN,China,2020,1410317500,1406585000,1412360000,6
DEU,Germany,1960,75497034,72814900,77909682,10
DEU,Germany,1970,78446317,78091820,78967433,10
DEU,Germany,1980,78115795,77684873,78751283,10


In [0]:
GOLD_TABLE = f"{CATALOG}.{SCHEMA}.population_gold"

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

print("Created:", GOLD_TABLE)

Created: workspace.default.population_gold


In [0]:
%sql
SELECT *
FROM population_gold
ORDER BY country_name, decade;

country_code,country_name,decade,avg_population,min_population,max_population,years_available
CHN,China,1960,714953000,660330000,796025000,10
CHN,China,1970,901944500,818315000,969005000,10
CHN,China,1980,1046603000,981235000,1118650000,10
CHN,China,1990,1196836000,1135185000,1252735000,10
CHN,China,2000,1298791000,1262645000,1331260000,10
CHN,China,2010,1374640000,1337705000,1407745000,10
CHN,China,2020,1410317500,1406585000,1412360000,6
DEU,Germany,1960,75497034,72814900,77909682,10
DEU,Germany,1970,78446317,78091820,78967433,10
DEU,Germany,1980,78115795,77684873,78751283,10


In [0]:
for table in [
    ICEBERG_TABLE,
    BRONZE_TABLE,
    SILVER_TABLE,
    GOLD_TABLE
]:
    print("\nTABLE:", table)

    spark.sql(
        f"DESCRIBE DETAIL {table}"
    ).select(
        "format",
        "name",
        "numFiles",
        "sizeInBytes"
    ).show(truncate=False)


TABLE: workspace.default.wb_population_landing_iceberg
+-------+-----------------------------------------------+--------+-----------+
|format |name                                           |numFiles|sizeInBytes|
+-------+-----------------------------------------------+--------+-----------+
|iceberg|workspace.default.wb_population_landing_iceberg|1       |6084       |
+-------+-----------------------------------------------+--------+-----------+


TABLE: workspace.default.population_bronze
+------+-----------------------------------+--------+-----------+
|format|name                               |numFiles|sizeInBytes|
+------+-----------------------------------+--------+-----------+
|delta |workspace.default.population_bronze|1       |5659       |
+------+-----------------------------------+--------+-----------+


TABLE: workspace.default.population_silver
+------+-----------------------------------+--------+-----------+
|format|name                               |numFiles|sizeInByte